# ✨ VibeCast — Creative Commentary Studio
## Complete Instructor Notebook: Groq + one ElevenLabs v4 voice + one animated character

VibeCast turns unrelated words into an original spoken mini-scene. The user enters ideas such as `Umbrella, Fall, Red, Gucci, Fun`, selects one of five vibes, and the **same single character** performs the generated commentary.

### The design is intentionally simple
- There is **one character on screen**.
- There is **one ElevenLabs voice ID**.
- There is **no host selector, no sidekick and no second speaker**.
- The selected vibe—Energetic, Dramatic, Funny, Emotional or Professional—changes the writing direction and the way the one character performs the narration.
- The character's visual identity and speaker identity stay the same in every generation.

### Main technologies
- **Groq:** creates commentary from random words and phrases.
- **ElevenLabs v4:** narrates the script using one voice ID and vibe-specific audio tags.
- **Streamlit:** provides the interactive website.
- **Google Colab:** runs the project and exposes the website through an iframe.

### Account limitation to remember
Your ElevenLabs account previously returned `detected_unusual_activity` and said Free Tier API access had been disabled. A single voice is the correct simpler project design, but it cannot bypass that provider-side restriction. If it remains active, live TTS requests will still fail. Contact ElevenLabs support or use an eligible account/plan; do not create extra accounts or rotate networks to evade the restriction.

Official references:
- Eleven v4 audio tags: https://elevenlabs.io/docs/help-center/product/core-capabilities/text-to-speech/how-do-audio-tags-work-with-eleven-v3-and-v4
- Text-to-speech API: https://elevenlabs.io/docs/api-reference/text-to-speech/convert
- Account unusual-activity restriction: https://elevenlabs.io/docs/overview/administration/account


## Learning map

By the end of this notebook, you will understand how to:

1. Extract the complete VibeCast project and understand its files.
2. Install the lightweight project dependencies.
3. Configure Groq, ElevenLabs and **one** voice ID through Colab Secrets.
4. Understand the difference between an API key and a voice ID.
5. Generate creative commentary from unrelated keywords.
6. Use Groq prompts to make the writing match the selected vibe.
7. Add Eleven v4 audio tags so the **same character** performs each vibe differently.
8. Generate and listen to a real MP3 sample when ElevenLabs API access is available.
9. Compare five vibes while keeping the script and voice ID constant.
10. Run offline tests and launch the one-character Streamlit website.

**Key concept:** Character identity does not change. Only the selected vibe and performance intensity change the way the character speaks.


## 0. Upload and unpack the complete project ZIP

The instructor notebook is included inside the project ZIP. Run this cell to upload the latest `VibeCast_ElevenLabs_Project.zip` and extract its `VibeCast_Project` folder into the Colab runtime.

**Why do we do this?** Colab provides a temporary virtual machine. The notebook needs the actual Python files (`app.py` and `commentary_engine.py`) before it can import and test them.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import os, shutil

print("Upload VibeCast_ElevenLabs_Project.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise FileNotFoundError("Please upload the VibeCast_ElevenLabs_Project.zip file.")

PROJECT_DIR = Path("/content/VibeCast_Project")
old_process = globals().get("process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try:
        old_process.wait(timeout=5)
    except Exception:
        old_process.kill()
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

with ZipFile(zip_names[0]) as archive:
    archive.extractall("/content")

required = ["app.py", "commentary_engine.py", "requirements.txt", "tests/test_engine.py"]
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Project archive is missing: {missing}")
os.chdir(PROJECT_DIR)
print("Project extracted to:", PROJECT_DIR)
print("Required files found:", required)

## 1. Install the project dependencies

VibeCast deliberately calls ElevenLabs through its documented HTTP REST endpoint using `requests`. This keeps the dependency list small and avoids adding a heavy local TTS model or a GPU requirement.

The requirements include:

- `streamlit` — website interface.
- `groq` — text generation.
- `requests` — secure HTTP calls to ElevenLabs.
- `python-dotenv` — load private keys and voice IDs from a local `.env` file.

**No Gemini package, Qwen package, Transformers, Diffusers, or local TTS model is needed.**

In [ ]:
%pip install -q -r requirements.txt
print("VibeCast dependencies installed.")

## 2. Import libraries and inspect the project

We import the standard libraries and the VibeCast engine. The engine is the reusable backend: the notebook and the Streamlit website both call the same functions.

In [ ]:
import os
import sys
import json
import time
import base64
import requests
from pathlib import Path
from IPython.display import display, Audio, Markdown
from dotenv import load_dotenv

import commentary_engine as engine

print("Groq model:", engine.GROQ_MODEL)
print("ElevenLabs model:", engine.ELEVENLABS_TTS_MODEL)
print("Character:", engine.CHARACTER_NAME)
print("Available vibes:", engine.STYLES)
print("Available languages:", engine.LANGUAGES)


## 3. Configure API keys and the one voice ID safely

Open the **Secrets** panel in Colab and add exactly these three entries:

- `GROQ_API_KEY` — generates the written commentary.
- `ELEVENLABS_API_KEY` — authenticates TTS requests.
- `ELEVENLABS_VOICE_ID` — selects the **one voice** used by the one VibeCast character.

Enable notebook access for all three secret values. The code below retrieves the values without displaying them. Never paste a real API key into a notebook cell or publish it in a repository.


In [ ]:
from google.colab import userdata

for secret_name in ("GROQ_API_KEY", "ELEVENLABS_API_KEY", "ELEVENLABS_VOICE_ID"):
    try:
        secret_value = userdata.get(secret_name) or ""
    except Exception:
        secret_value = ""
    if secret_value:
        os.environ[secret_name] = secret_value.strip()
    else:
        os.environ.pop(secret_name, None)

print("Groq key configured:", bool(os.getenv("GROQ_API_KEY")))
print("ElevenLabs key configured:", bool(os.getenv("ELEVENLABS_API_KEY")))
print("Single shared voice ID configured:", bool(os.getenv("ELEVENLABS_VOICE_ID")))
print("Secret values were not displayed.")


## 4. Check local configuration before calling ElevenLabs

The account/subscription endpoint requires the separate `user_read` permission and is not necessary to create speech. To avoid an unnecessary permission requirement, this section only checks whether the key and one voice ID are present. The real API test is Section 17, which makes a short text-to-speech request.

**Important:** Your previously reported provider response `detected_unusual_activity` means ElevenLabs has disabled Free Tier API use for your account. If that restriction is still active, Section 17 will fail even when the API key and voice ID are correct. Changing to one voice does not bypass it.


In [ ]:
required_values = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY", "").strip(),
    "ELEVENLABS_API_KEY": os.getenv("ELEVENLABS_API_KEY", "").strip(),
    "ELEVENLABS_VOICE_ID": os.getenv("ELEVENLABS_VOICE_ID", "").strip(),
}

for name, value in required_values.items():
    print(f"{name}: {'configured' if value else 'MISSING'}")

if not required_values["ELEVENLABS_API_KEY"]:
    print("Add ELEVENLABS_API_KEY in Colab Secrets and rerun Section 3.")
if not required_values["ELEVENLABS_VOICE_ID"]:
    print("Add one API-accessible voice ID as ELEVENLABS_VOICE_ID.")


## 5. Configure the one VibeCast character voice

VibeCast has one recurring on-screen character and one narrator voice. There are no named host profiles, alternate speakers or sidekick characters.

### Where to get the voice ID
1. Sign in to your ElevenLabs account.
2. Open a voice that your account is permitted to use through API text-to-speech.
3. Open its details/options and copy its voice ID.
4. Add that ID to Colab Secrets with the exact name `ELEVENLABS_VOICE_ID`.
5. Rerun Section 3 so the notebook loads it.

This project does **not** call Voice Design or voice-creation endpoints. It only uses an existing voice ID with the text-to-speech endpoint. Your recent `detected_unusual_activity` message means the ElevenLabs provider may still reject TTS calls even after the ID is configured; this is an account restriction, not a notebook configuration error.

The same speaker is retained for every vibe. Energetic, Dramatic, Funny, Emotional and Professional change the acting style—not the character or voice identity.


### 5A. Load and validate the one shared voice ID

This cell deliberately makes no API calls and spends no credits. It checks only that the shared voice ID has been added to the environment.


In [ ]:
SHARED_VOICE_ID = os.getenv("ELEVENLABS_VOICE_ID", "").strip()

if not SHARED_VOICE_ID:
    print("Voice ID missing. Add ELEVENLABS_VOICE_ID to Colab Secrets, enable access, and rerun Section 3.")
else:
    print("One VibeCast character voice ID is configured.")
    print("The ID is hidden and will be used for every selected vibe.")


## 6. Save the one voice ID into the extracted project's `.env`

The Streamlit application reads `.env` when running locally. This cell writes the three keys plus the **single** voice ID into the temporary Colab project. Secret values are never printed.


In [ ]:
from dotenv import set_key

env_path = PROJECT_DIR / ".env"
if not env_path.exists():
    env_path.write_text("", encoding="utf-8")

values_to_save = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY", ""),
    "ELEVENLABS_API_KEY": os.getenv("ELEVENLABS_API_KEY", ""),
    "ELEVENLABS_TTS_MODEL": "eleven_v4",
    "ELEVENLABS_OUTPUT_FORMAT": "mp3_44100_128",
    "ELEVENLABS_VOICE_ID": os.getenv("ELEVENLABS_VOICE_ID", ""),
}
for key, value in values_to_save.items():
    if value:
        set_key(str(env_path), key, value)

load_dotenv(env_path, override=True)
print("Project .env updated. Secret values were not printed.")


## 7. Verify the one character configuration

This is a local check. It confirms that the app has exactly one voice ID setting. It does not contact ElevenLabs and does not consume TTS credits.


In [ ]:
shared_voice_id = os.getenv("ELEVENLABS_VOICE_ID", "").strip()
print("Single character:", engine.CHARACTER_NAME)
print("One ElevenLabs voice ID configured:", bool(shared_voice_id))
print("Number of configured speaker IDs:", 1 if shared_voice_id else 0)


## 8. Reload the commentary engine and inspect the one-voice configuration

The notebook and website import the same `commentary_engine.py`. Reload it after saving `.env` so the current runtime uses the updated one-voice configuration.


In [ ]:
import importlib
import commentary_engine as engine
engine = importlib.reload(engine)

print("TTS model:", engine.ELEVENLABS_TTS_MODEL)
print("Character:", engine.CHARACTER_NAME)
print("One voice ID configured:", bool(engine.elevenlabs_voice_id()))
print("Available vibes:", ", ".join(engine.STYLES))


## 9. Keyword parsing — clean the user's ideas

The user can enter words separated by commas, semicolons, vertical bars, or new lines. `parse_keywords()` removes empty entries and duplicate words while preserving the spelling the user supplied.

This is important because the AI should receive a clean list of creative ingredients rather than a single unstructured string.

In [ ]:
sample_input = "Umbrella, Fall\nRed; Gucci | Fun, umbrella"
keywords = engine.parse_keywords(sample_input)
print("Original input:", sample_input)
print("Clean keywords:", keywords)
print("Unique ideas:", len(keywords))

## 10. Validate the user's input

The website accepts 2–18 unique ideas. The lower limit ensures there is something to connect; the upper limit keeps the generated scene focused and avoids unnecessary API usage.

In [ ]:
def validate_ideas(raw_input):
    items = engine.parse_keywords(raw_input)
    if len(items) < 2:
        return False, "Add at least two words or short phrases."
    if len(items) > 18:
        return False, "Use 18 ideas or fewer."
    return True, f"{len(items)} ideas are ready."

for sample in ["Umbrella", "Umbrella, Fall", "a, b, c"]:
    print(sample, "→", validate_ideas(sample))

## 11. Prompt engineering — tell Groq what to create

A system prompt defines the AI's job and its rules. VibeCast asks Groq to connect unrelated ideas into one original spoken scene. The prompt includes the selected vibe, language, commentary length and performance intensity.

There is only one recurring character. The prompt explicitly tells Groq not to introduce a co-host, sidekick, second speaker or dialogue exchange.

The five vibes affect the **writing** as well as the later voice performance. For example, a Dramatic script builds suspense, while a Funny script uses playful comparisons and comic timing.


In [ ]:
example_prompt = engine.build_system_prompt(
    style="Dramatic",
    language="English",
    commentator_name=engine.CHARACTER_NAME,
    length="Standard",
    intensity=5,
)
print(example_prompt)


## 12. Offline commentary — test the concept without Groq

`offline_commentary()` is a deterministic fallback. It is not another AI model. It shows how the application can still create a simple result if Groq is unavailable.

In [ ]:
words = ["Umbrella", "Fall", "Red", "Gucci", "Fun"]
for vibe in engine.STYLES:
    print("\nVIBE:", vibe)
    print(engine.offline_commentary(words, vibe, "English", "Standard"))

## 13. Generate commentary with Groq

This cell attempts a real Groq request when `GROQ_API_KEY` is configured. If the key is missing or Groq is unreachable, the engine returns its clearly labelled offline fallback.

In [ ]:
commentary, generation_status, used_fallback, result_label = engine.generate_commentary(
    words="Umbrella, Fall, Red, Gucci, Fun",
    style="Dramatic",
    language="English",
    commentator_name=engine.CHARACTER_NAME,
    length="Standard",
    intensity=5,
)

print("Status:", generation_status)
print("Used fallback:", used_fallback)
display(Markdown("### Generated commentary\n\n" + commentary))


## 14. Understand the two voice controls

There is one character and one speaker voice. The app offers two controls for its delivery:

- **Vibe:** Energetic, Dramatic, Funny, Emotional or Professional. This selects the main emotional acting direction and the Eleven v4 audio tags.
- **Intensity:** 1–5. This controls how strongly the selected performance is expressed and slightly adjusts Eleven v4 stability.

The selected vibe also changes Groq's writing direction. For the voice test, keep the script and voice ID constant and change only the vibe. That lets you hear the delivery difference without changing the character.


## 15. Audio tags — instruct Eleven v4 how to perform the script

Eleven v4 supports inline audio tags written in square brackets. These tags are sent as part of the TTS text field; they are not added to the visible commentary shown in the app.

Examples documented by ElevenLabs include `[curious]`, `[mischievously]`, `[whispers]`, `[shouts]`, `[laughs]`, `[clears throat]`, and `[sighs]`. The engine uses a small number of tags at meaningful sentence boundaries rather than decorating every word. For Eleven v4, pauses should be guided with punctuation and ellipses; do not use SSML `<break>` tags.

In [ ]:
sample_script = (
    "The rain was falling. A red umbrella appeared in the middle of the street. "
    "And then, Gucci arrived. What a dramatic entrance!"
)

for vibe in engine.STYLES:
    tagged = engine.decorate_elevenlabs_transcript(
        sample_script, vibe, intensity=5
    )
    print("\n", vibe, "\n", tagged)


## 16. Inspect the one character and performance settings before spending credits

This cell displays the one character name, whether the single voice ID is configured, and the director note for the selected vibe. It does not contact ElevenLabs. The real v4 request uses Stability and Similarity plus inline audio tags; v4 does not support the older Style and Speed sliders.


In [ ]:
selected_vibe = "Dramatic"
selected_intensity = 5

print("Character:", engine.CHARACTER_NAME)
print("One voice ID configured:", bool(engine.elevenlabs_voice_id()))
print("Selected vibe:", selected_vibe)
print("Intensity:", selected_intensity)
print("\nDirector note (inspection only):")
print(engine.build_tts_prompt(
    style=selected_vibe,
    language="English",
    intensity=selected_intensity,
))
print("\nTagged transcript:")
print(engine.decorate_elevenlabs_transcript(
    sample_script, selected_vibe, selected_intensity
))


## 17. Generate one real ElevenLabs v4 audio sample

**This cell makes a real API request and uses credits.** It needs:

- `ELEVENLABS_API_KEY`
- the one shared voice ID
- text within the model's request limit

The response from the create-speech endpoint is MP3 bytes. The engine returns those bytes and a status label, which the app can use for the audio player and download button.

In [ ]:
if not os.getenv("ELEVENLABS_API_KEY"):
    print("Add ELEVENLABS_API_KEY in Colab Secrets first.")
elif not engine.elevenlabs_voice_id():
    print("The one character voice ID is missing. Add ELEVENLABS_VOICE_ID in Colab Secrets and rerun Section 3.")
else:
    sample_audio, sample_voice_status = engine.synthesize_expressive_audio(
        text=sample_script,
        style=selected_vibe,
        language="English",
        intensity=selected_intensity,
    )
    print(sample_voice_status)
    if sample_audio:
        audio_path = PROJECT_DIR / "vibecast_sample.mp3"
        audio_path.write_bytes(sample_audio)
        display(Audio(sample_audio, autoplay=False))
        print("Saved MP3:", audio_path)
    else:
        print("TTS did not return audio. Read the status above before continuing.")


## 18. A/B comparison — one character, one voice, five different vibes

This is the key experiment. Keep the **same script, same character and same voice ID**. Only change the vibe. Listen to each result and compare how the one speaker performs the words.

This sends five TTS requests and uses credits. The sample is intentionally short. If you want to conserve credits, run only the one-sample test in Section 17 first.


In [ ]:
AB_SCRIPT = "The rain was falling. A red umbrella appeared. Then Gucci arrived, and the entire street turned to look. What a moment!"

if not os.getenv("ELEVENLABS_API_KEY") or not os.getenv("ELEVENLABS_VOICE_ID"):
    print("Configure the ElevenLabs API key and the one voice ID before running this comparison.")
else:
    vibe_audio_results = {}
    for vibe in engine.STYLES:
        print("\nGenerating:", vibe)
        audio_bytes, status = engine.synthesize_expressive_audio(
            text=AB_SCRIPT,
            style=vibe,
            language="English",
            intensity=5,
        )
        print(status)
        if audio_bytes:
            vibe_audio_results[vibe] = audio_bytes
            display(Audio(audio_bytes, autoplay=False))


## 19. A/B comparison — one character, same vibe, three intensity levels

Keep the script, speaker voice and vibe fixed. Change only intensity (2, 3 and 5). This helps you decide how expressive the same character should sound without changing its identity.

This sends three TTS requests and uses credits. Run only if the one-sample and five-vibe tests have already worked.


In [ ]:
INTENSITY_SCRIPT = "Everything was quiet. Then the red umbrella appeared, and the whole scene changed in an instant."
INTENSITY_VIBE = "Dramatic"

if not os.getenv("ELEVENLABS_API_KEY") or not os.getenv("ELEVENLABS_VOICE_ID"):
    print("Configure the ElevenLabs API key and the one voice ID before running this comparison.")
else:
    intensity_audio_results = {}
    for level in (2, 3, 5):
        print("\nCharacter intensity:", level)
        audio_bytes, status = engine.synthesize_expressive_audio(
            text=INTENSITY_SCRIPT,
            style=INTENSITY_VIBE,
            language="English",
            intensity=level,
        )
        print(status)
        if audio_bytes:
            intensity_audio_results[level] = audio_bytes
            display(Audio(audio_bytes, autoplay=False))


## 20. Understand language support and pronunciation

ElevenLabs v4 is documented for 90+ languages, but pronunciation quality depends on the selected voice and the language. VibeCast keeps English, Urdu and Roman Urdu as writing-language options. Test your chosen voice with actual Urdu and Roman Urdu phrases before relying on it. Roman Urdu is informal Latin-script Urdu, so pronunciation can vary with spelling. If a brand, person or uncommon word is pronounced incorrectly, adjust the written script and listen again.

The browser fallback is separate from ElevenLabs. If API audio is unavailable, the browser uses voices installed on the user's device; quality and language coverage vary.


## 21. Inspect important project files

The project folder contains:

- `app.py` — Streamlit controls, light theme, one-character performance stage, downloads and session history.
- `commentary_engine.py` — keyword parsing, Groq prompt and commentary generation, vibe-specific audio tags and ElevenLabs request.
- `requirements.txt` — small Python dependency list.
- `.env.example` — placeholder names for API keys and one voice ID.
- `assets/host.svg` — the only character artwork, with an animated mouth.
- `tests/test_engine.py` — offline tests using mocked API responses.

Both the notebook and website use `commentary_engine.py`, so there is only one place where the actual voice-generation request is implemented.


In [ ]:
for file_name in [
    "app.py", "commentary_engine.py", "requirements.txt", ".env.example",
    "assets/host.svg", "tests/test_engine.py"
]:
    path = PROJECT_DIR / file_name
    print(f"{file_name:30} exists={path.exists()} size={(path.stat().st_size if path.exists() else 0):>8} bytes")


## 22. Run automated offline tests

The tests do not contact Groq or ElevenLabs and do not use API credits. They check:

- keyword parsing, deduplication and validation;
- creative fallback output for all five vibes;
- Groq prompt rules that require exactly one character;
- different Eleven v4 audio tags for the five vibes;
- the exact v4 request model, one voice endpoint and output format;
- the same character voice ID is used for every vibe;
- missing key/voice ID handling;
- account unusual-activity error handling, empty responses and text-length validation.


In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=str(PROJECT_DIR),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("Test exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("One or more offline tests failed. Read the output above.")

## 23. Launch VibeCast in Google Colab

The website runs as a Streamlit server on port `8501`. Colab exposes it in the notebook through `serve_kernel_port_as_iframe()`.

Before launching, confirm that:

1. `GROQ_API_KEY` is configured if you want AI-written commentary.
2. `ELEVENLABS_API_KEY` is configured.
3. Exactly one `ELEVENLABS_VOICE_ID` is configured.
4. The project offline tests pass.
5. The ElevenLabs account is allowed to use API text-to-speech. If the provider returns `detected_unusual_activity`, the account restriction must be resolved by ElevenLabs; the app cannot bypass it.


In [ ]:
import subprocess, time, sys
from google.colab import output

log_path = Path("/content/vibecast_streamlit.log")
old_process = globals().get("process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try:
        old_process.wait(timeout=5)
    except Exception:
        old_process.kill()

log_file = open(log_path, "w")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    cwd=str(PROJECT_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if process.poll() is not None:
    print(log_path.read_text(errors="ignore")[-8000:])
    raise RuntimeError("Streamlit did not start. Read the log output above.")

print("VibeCast is running on port 8501.")
output.serve_kernel_port_as_iframe(8501, height=900)

## 24. Troubleshooting guide

| Symptom | Likely cause | What to do |
|---|---|---|
| `ELEVENLABS_API_KEY` missing | Colab Secret was not added or notebook access is off | Add the secret, enable access, rerun Section 3 |
| `ELEVENLABS_VOICE_ID` missing | No shared voice ID configured | Add one API-accessible voice ID to Colab Secrets |
| HTTP 401 / invalid key | Key is wrong, revoked or malformed | Create/update an API key and check its text-to-speech permission |
| HTTP 401 `detected_unusual_activity` | ElevenLabs disabled Free Tier API access on the account | One voice ID does not bypass this. Contact ElevenLabs support or use an eligible paid subscription |
| HTTP 401 `missing_permissions` | API key lacks a permission | Edit the API key permissions in ElevenLabs settings |
| HTTP 403 `feature_not_available` on Voice Design | Voice creation through API is paid-only | VibeCast does not call Voice Design; use one existing, API-accessible voice ID |
| HTTP 403 on TTS | Account/voice/model access restriction | Check API entitlement and account status |
| HTTP 404 | Incorrect voice ID | Copy the voice ID from an account voice available for API use |
| HTTP 422 | Invalid request fields or unsupported text | Check model ID `eleven_v4`, audio tags and the 10,000-character limit |
| HTTP 429 | Credits or concurrency limit reached | Check account usage and retry when eligible |
| Browser voice sounds identical | Device speech fallback selected | This is browser speech, not ElevenLabs. Device voices vary |

Do not create extra accounts or rotate keys to evade a Free Tier restriction. The account restriction must be resolved with the provider.


## 25. Free-plan usage and account restrictions

Plan allowances and API availability can change. Check the official pricing page before relying on free usage. Some API features (including voice creation through API) may require a paid subscription. The public Voice Library is not available through the API to all free-tier users.

**This notebook's current known blocker:** the account previously returned `detected_unusual_activity` and said Free Tier access was disabled. That is an account-level restriction. A shared voice ID simplifies configuration but does not restore blocked API access. Contact ElevenLabs support or use an eligible paid plan. Do not try to bypass the restriction by making extra accounts or rotating network identities.

VibeCast uses only one voice ID and does not create or clone voices through the API.


## 26. Run locally on Windows (optional)

1. Extract `VibeCast_SingleCharacter_ElevenLabs_Project.zip`.
2. Open PowerShell inside the `VibeCast_Project` folder.
3. Create and activate a virtual environment:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

4. Install dependencies:

```powershell
pip install -r requirements.txt
```

5. Copy `.env.example` to `.env` and set `GROQ_API_KEY`, `ELEVENLABS_API_KEY`, and one `ELEVENLABS_VOICE_ID`.
6. Launch:

```powershell
streamlit run app.py
```


## 27. Customization challenges

1. Improve the five vibe tag combinations while retaining one speaker and one character.
2. Add carefully chosen audio tags at sentence boundaries for more natural emotional progression.
3. Compare stability values and document which settings sound expressive without becoming erratic.
4. Test English, Urdu and Roman Urdu separately; pronunciation quality varies by voice and language.
5. Improve the single character's idle, thinking and speaking animations without adding another character.
6. Add clearer account-error messages without ever displaying API keys.

Keep the same character and speaker identity. Only the selected vibe and intensity should change the performance.


## 28. Final checklist

- [ ] The project ZIP extracted to `/content/VibeCast_Project`.
- [ ] Dependencies installed; no local Qwen, Transformers or Diffusers model is required.
- [ ] Groq and ElevenLabs API keys loaded from Colab Secrets.
- [ ] Exactly one API-accessible voice ID configured as `ELEVENLABS_VOICE_ID`.
- [ ] The interface shows one character only—no sidekick or host selector.
- [ ] One real ElevenLabs v4 MP3 sample generated and listened to (requires active account API access).
- [ ] The same voice was tested with all five vibes.
- [ ] Intensity comparison completed (optional; uses extra credits).
- [ ] All offline tests passed.
- [ ] Streamlit opened through the Colab iframe.

**Final architecture:** random keywords → Groq creative commentary → one ElevenLabs v4 voice ID + selected vibe audio tags → MP3 → one animated VibeCast character.
